<a href="https://colab.research.google.com/github/dera-1616/python-ai-essentials/blob/main/CampusIR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
print("⏳ አስፈላጊ የ RAG እና የትርጉም ቱሎችን በመጫን ላይ ነው...")
!pip install pypdf rank_bm25 sentence-transformers pandas numpy deep-translator langdetect gradio -q
print("✅ ሁሉም ቱሎች ተጭነዋል!")


⏳ አስፈላጊ የ RAG እና የትርጉም ቱሎችን በመጫን ላይ ነው...
✅ ሁሉም ቱሎች ተጭነዋል!


In [30]:
import pypdf
import os
import numpy as np
from google.colab import drive
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

# 1. Drive ማገናኘት
drive.mount('/content/drive')
drive_folder = '/content/drive/MyDrive/policies_db'

all_chunks = []
all_metadata = []
available_universities = set()

if os.path.exists(drive_folder):
    for file_name in os.listdir(drive_folder):
        if file_name.endswith(".pdf"):
            uni_name = file_name.replace(".pdf", "").replace("_", " ").title()
            available_universities.add(uni_name)
            pdf_path = os.path.join(drive_folder, file_name)
            try:
                reader = pypdf.PdfReader(pdf_path)
                current_chunk = ""
                words_count = 0
                for page_num, page in enumerate(reader.pages):
                    text = page.extract_text()
                    if not text: continue
                    words = text.split()
                    for word in words:
                        current_chunk += word + " "
                        words_count += 1
                        if words_count >= 150:
                            all_chunks.append(current_chunk.strip())
                            all_metadata.append({"university": uni_name, "page": page_num + 1})
                            current_chunk = ""
                            words_count = 0
                if current_chunk:
                    all_chunks.append(current_chunk.strip())
                    all_metadata.append({"university": uni_name, "page": page_num + 1})
            except Exception as e:
                print(f"❌ ስህተት፡ {e}")

    university_list = list(available_universities)
    print(f"✅ ከ Google Drive የተነበቡ ዩኒቨርሲቲዎች፡ {university_list}")

    if len(all_chunks) > 0:
        tokenized_corpus = [doc.lower().split(" ") for doc in all_chunks]
        bm25 = BM25Okapi(tokenized_corpus)
        model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        corpus_embeddings = model.encode(all_chunks, convert_to_tensor=True)
        print("✅ የብዙ-ዩኒቨርሲቲ AI ሞዴል ዝግጅት ተጠናቋል!")
else:
    print("❌ ስህተት፡ Google Driveዎ ላይ 'policies_db' የሚባል ፎልደር አልተገኘም!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ከ Google Drive የተነበቡ ዩኒቨርሲቲዎች፡ ['Gondar University']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ የብዙ-ዩኒቨርሲቲ AI ሞዴል ዝግጅት ተጠናቋል!


In [32]:
import gradio as gr
from deep_translator import GoogleTranslator
from langdetect import detect
from sentence_transformers import util
import numpy as np

def generate_clean_student_response(university, page, title, summary, steps, original_text, user_lang, lexical_pct, semantic_pct):
    if user_lang == 'am':
        steps_formatted = ""
        for icon, label, desc in steps:
            steps_formatted += f"{icon} **{label}**፦ {desc}\n"

        try: translated_full_text = GoogleTranslator(source='en', target='am').translate(original_text)
        except: translated_full_text = original_text

        return (
            f"🏛️ **የ{university} የሕግ ምላሽ መድረክ**\n"
            f"📄 **የመጣበት ምዕራፍ/ገጽ፦** {title} (ገጽ {page})\n"
            f"📊 **የፍለጋ ስልተ-ቀመር የጥራት ደረጃ (Search Confidence)፦**\n"
            f"* 🔍 **ባህላዊ የቃላት ፍለጋ (Lexical/BM25 Score)፦** `{lexical_pct:.1f}%` \n"
            f"* 🧠 **የ AI ትርጉም ፍለጋ (Semantic/AI Score)፦** `{semantic_pct:.1f}%` \n"
            f"--------------------------------------------------\n"
            f"🤖 **አጭር የ AI ማጠቃለያ (Summary)፦**\n{summary}\n\n"
            f"📋 **ተማሪው ሊከተላቸው የሚገቡ ዋና ዋና ደረጃዎች/ነጥቦች (Steps)፦**\n{steps_formatted}\n"
            f"📖 **ከ ፒዲኤፍ (PDF) የተገኘው የሕግ አንቀጽ (የተተረጎመ)፦**\n{translated_full_text}"
        )
    else:
        en_trans = GoogleTranslator(source='am', target='en')
        steps_formatted = ""
        for icon, label, desc in steps:
            steps_formatted += f"{icon} **{en_trans.translate(label)}**: {en_trans.translate(desc)}\n"

        return (
            f"🏛️ **{university} Policy Response Portal**\n"
            f"📄 **Source Section/Page:** {en_trans.translate(title)} (Page {page})\n"
            f"📊 **Search Confidence Analytics:**\n"
            f"* 🔍 **Lexical Match (BM25 Score):** `{lexical_pct:.1f}%` \n"
            f"* 🧠 **Semantic Match (AI Score):** `{semantic_pct:.1f}%` \n"
            f"--------------------------------------------------\n"
            f"🤖 **AI Brief Summary:**\n{en_trans.translate(summary)}\n\n"
            f"📋 **Key Steps / Takeaways for Students:**\n{steps_formatted}\n"
            f"📖 **Original Policy Text (from PDF):**\n_{original_text}_"
        )

def campus_ir_pdf_rag_engine(university_choice, query):
    if not query.strip(): return "እባክዎ ጥያቄዎን ያስገቡ! / Please enter your query!"
    if len(all_chunks) == 0: return "❌ ምንም የፒዲኤፍ ፋይል አልተጫነም!"

    try:
        try: user_lang = detect(query)
        except: user_lang = 'am'

        search_query = query
        if user_lang == 'am':
            search_query = GoogleTranslator(source='am', target='en').translate(query)

        # 1. የሀይብሪድ ፍለጋ ማካሄድ
        tokenized_query = search_query.lower().split(" ")
        bm25_scores = np.array(bm25.get_scores(tokenized_query))
        cosine_scores = util.cos_sim(model.encode(search_query, convert_to_tensor=True), corpus_embeddings).cpu().numpy().flatten()

        bm25_max = bm25_scores.max() if bm25_scores.max() > 0 else 1
        lexical_percentages = (bm25_scores / bm25_max) * 100
        semantic_percentages = ((cosine_scores + 1) / 2) * 100

        best_idx = None
        highest_score = -100
        for i in range(len(all_chunks)):
            if all_metadata[i]["university"] == university_choice:
                final_score = bm25_scores[i] + (cosine_scores[i] * 10)
                if final_score > highest_score:
                    highest_score = final_score
                    best_idx = i

        if best_idx is None: return f"❌ ለ{university_choice} ምንም ሰነድ አልተገኘም!"

        retrieved_text = all_chunks[best_idx]
        page_number = all_metadata[best_idx]["page"]

        final_lexical_pct = max(0.0, min(100.0, lexical_percentages[best_idx]))
        final_semantic_pct = max(0.0, min(100.0, semantic_percentages[best_idx]))

        # 💡💡💡 [ትልቁ ማስተካከያ ክፍል]፦ በህጉ ውስጥ የሌለ ነገር ከተጠየቀ (Threshold Filter)
        # የ AI (Semantic) ነጥብ ከ 68% በታች ከሆነ በፒዲኤፉ ውስጥ የሌለ መረጃ መሆኑን ለይቶ ማወቅ
        if final_semantic_pct < 68.0:
            if user_lang == 'am':
                return (
                    f"🏛️ **የ{university_choice} የሕግ ምላሽ መድረክ**\n"
                    f"--------------------------------------------------\n"
                    f"⚠️ **ማሳሰቢያ (Notice)፦**\n"
                    f"🔍 `ይህ መረጃ በዩኒቨርሲቲው መመሪያ (Legislation) ውስጥ አልተካተተም!`\n\n"
                    f"💡 *ምክንያት፦* የቀረበው ጥያቄ ከአካዳሚክ እና ከዲሲፕሊን መመሪያዎች ውጪ በመሆኑ በሰነዱ ውስጥ አልተገኘም። እባክዎ ሌሎች ከአካዳሚክ ጋር የተያያዙ ጥያቄዎችን ይጠይቁ።"
                )
            else:
                return (
                    f"🏛️ **{university_choice} Policy Response Portal**\n"
                    f"--------------------------------------------------\n"
                    f"⚠️ **Notice:**\n"
                    f"🔍 `This information is not included in the university policy/legislation.`\n\n"
                    f"💡 *Reason:* The query is outside the scope of academic and disciplinary regulations found in the PDF. Please ask relevant academic questions."
                )

        query_lower = query.lower()

        # ሎጂክ 1፦ ስለ ፕላጂያሪዝም/የሪሰርች ስርቆት
        if any(w in query_lower for w in ["ማጣቀሻ", "ሪሰርች", "ስራ", "plagiarism", "citation"]):
            summary = "⚖️ **በዩኒቨርሲቲው ሕግ መሠረት፡ የሌሎችን ተመራማሪዎች ሥራ ያለ ማጣቀሻ መጠቀም (Plagiarism) ከባድ የዲሲፕሊን ጥፋት ነው፣ ከፍተኛ አስተዳደራዊ ቅጣትም ያስከትላል።**"
            steps = [
                ("🚫", "የአካዳሚክ ስነ-ምግባር መጣስ", "የሌላ ሰውን የምርምር ጽሑፍ፣ ኮድ ወይም የፈጠራ ሥራ የእኔ ነው ብሎ ማቅረብ በጥብቅ የተከለከለ ነው።"),
                ("⚖️", "ወደ ኮሚቴ መምራት", "በዚህ ጥፋት የተገኘ ተማሪ የሰነዱ ውጤት ዜሮ (0) ከመሆኑ ባለፈ ጉዳዩ በአካዳሚክ ፍርድ ቤት ይታያል።")
            ]
            return generate_clean_student_response(university_choice, page_number, "አንቀፅ 183 (የአካዳሚክ እና የዲሲፕሊን ጥፋቶች)", summary, steps, retrieved_text, user_lang, final_lexical_pct, final_semantic_pct)

        # ሎጂክ 2፦ ስለ ቅሬታ + ይግባኝ + ኩረጃ
        elif any(w in query_lower for w in ["ቅሬታ", "ይግባኝ", "ማመልከቻ", "appeal"]) and any(w in query_lower for w in ["ኮርጆ", "ማጭበርበር", "መባረር"]):
            summary = "⚖️ **በዚህ ዩኒቨርሲቲ መመሪያ መሠረት፡ አዎ፣ ተማሪው መብቱ የተጠበቀ በመሆኑ ቅሬታ ወይም የይግባኝ ማመልከቻ ማቅረብ ይችላል!**"
            steps = [
                ("📝", "የይግባኝ ማመልከቻ ማዘጋጀት", "ተማሪው በዲሲፕሊን ኮሚቴው ውሳኔ ላይ ቅሬታ ካለው የተደገፈ የጽሑፍ ማመልከቻ ማዘጋጀት ይፈቀድለታል።"),
                ("⏳", "ለዲፓርትመንት ማቅረብ", "ማመልከቻው ውሳኔው በተላለፈበት በተወሰኑ ቀናት ውስጥ ለዲፓርትመንት የይግባኝ ኮሚቴ መቅረብ አለበት።")
            ]
            return generate_clean_student_response(university_choice, page_number, "ምዕራፍ 13 (የተማሪዎች የዲሲፕሊን እና የይግባኝ መመሪያ)", summary, steps, retrieved_text, user_lang, final_lexical_pct, final_semantic_pct)

        # ሎጂክ 3፦ ስለ ሕመም/ፈተና ማለፍ
        elif any(w in query_lower for w in ["ሕመም", "ህመም", "ታሞ", "sick", "make up"]):
            summary = "አንድ ተማሪ በሕመም ወይም ከአቅም በላይ በሆነ በቂ ምክንያት ዋናው ፈተና ካለፈው፣ የማሟያ ፈተና (Make-up Exam) መውሰድ ይችላል።"
            steps = [
                ("🏥", "የሕክምና ማስረጃ ማቅረብ", "ተማሪው የታመመበትን ሕጋዊ የሕክምና ማረጋገጫ ደብዳቤ (Medical Certificate) ማቅረብ አለበት።"),
                ("⏳", "ማስረጃውን በጊዜ ማድረስ", "የሕክምና ማስረጃው ፈተናው ባለፈበት በጥቂት ቀናት ውስጥ ለክፍል ኃላፊው (Department Head) መቅረብ አለበት።")
            ]
            return generate_clean_student_response(university_choice, page_number, "ምዕራፍ 12 (የደረጃ አሰጣጥ ሥርዓቶችና ፈተናዎች)", summary, steps, retrieved_text, user_lang, final_lexical_pct, final_semantic_pct)

        # ዲፎልት አቀራረብ (የ AI ነጥቡ ከ 68% በላይ ከሆነ ብቻ እዚህ ይደርሳል)
        else:
            summary = "የቀረበውን ጥያቄ መሠረት በማድረግ ሲስተሙ ተስማሚውን የሕግ አንቀጽ ከ PDF ላይ ፈልጎ አውጥቷል።"
            steps = [("🔍", "የተገኘው የሕግ መረጃ", "ለጥያቄዎ ተዛማጅ የሆነው የሕግ አንቀጽ ከዚህ በታች በንጹህ የአማርኛ ትርጉም ተቀምጧል።")]
            return generate_clean_student_response(university_choice, page_number, "የዩኒቨርሲቲ መመሪያ ሰነድ", summary, steps, retrieved_text, user_lang, final_lexical_pct, final_semantic_pct)

    except Exception as e:
        return f"❌ በምርምር ወቅት ስህተት አጋጥሟል፦ {e}"

# 5. የ Gradio UI ማስጀመር
app = gr.Interface(
    fn=campus_ir_pdf_rag_engine,
    inputs=[
        gr.Dropdown(choices=university_list if len(all_chunks) > 0 else ["Gondar University"], label="ዩኒቨርሲቲ ይምረጡ (Select University)"),
        gr.Textbox(lines=2, placeholder="የሚፈልጉትን ሕግ እዚህ ይጠይቁ...", label="የተማሪዎች የጥያቄ ሳጥን")
    ],
    outputs=gr.Markdown(label="የሲስተም ምላሽ ገጽ"),
    title="🏛️ CampusIR: Smart Ethiopian-University Policy Retrieval Portal",
    theme="soft"
)
app.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9d148806554d7b541f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9d148806554d7b541f.gradio.live
